# From Pilot to Payoff - 07: Conclusion & Summary

## Setup and data preparation

Reloads the data and rebuilds the features so this conclusion notebook can recompute the headline numbers on its own.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from statsmodels.formula.api import ols, logit

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [2]:
company = pd.read_csv('ai_company_adoption.csv')
country_index = pd.read_csv('country_ai_index.csv')
print(f'company: {company.shape[0]:,} x {company.shape[1]}    country index: {country_index.shape[0]} x {country_index.shape[1]}')
company.head(3)


company: 150,000 x 43    country index: 30 x 8


,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method
0,1,COMP-00001,2023,Q1,Italy,Europe,Education,Startup,57,48.31,...,2.65,1,1,3,2.52,9.45,53,5.20,WEF Survey,API Scrape
1,2,COMP-00001,2023,Q2,Italy,Europe,Education,Startup,57,48.31,...,5.77,2,2,5,4.77,0.00,51,6.98,McKinsey Report,Phone Interview
2,3,COMP-00001,2023,Q3,Italy,Europe,Education,Startup,57,48.31,...,6.94,3,3,2,12.87,9.74,40,4.12,Internal Corporate Survey,Research Compilation


In [3]:
# Feature engineering and merge (only what the recompute needs).
quarter_map = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
company = company.copy()
company['quarter_num'] = company['quarter'].map(quarter_map)
company['advanced_adoption'] = company['ai_adoption_stage'].isin(['partial', 'full']).astype(int)

# Log the skewed investment variable (positive amounts logged; rare zeros set to the median log).
_pos_investment = company['ai_investment_per_employee'] > 0
_log_investment = np.log(company['ai_investment_per_employee'].where(_pos_investment))
company['log_ai_investment_per_employee'] = _log_investment.fillna(_log_investment.median())

country_model = country_index.drop(columns=['region'])
country_model['log_gdp_per_capita'] = np.log1p(country_model['gdp_per_capita'])
country_model['log_ai_patent_filings_2024'] = np.log1p(country_model['ai_patent_filings_2024'])

df = company.merge(country_model, on='country', how='left')

# Latest observation per company (one row per firm) for the Bayesian estimate.
latest_obs = (df.sort_values(['company_id', 'survey_year', 'quarter_num'])
                .groupby('company_id', as_index=False)
                .tail(1)
                .reset_index(drop=True))

print(f'Merged data: {df.shape[0]:,} rows x {df.shape[1]:,} columns;  latest_obs: {latest_obs.shape[0]:,} rows')


Merged data: 150,000 rows x 54 columns;  latest_obs: 10,000 rows


In [4]:
# Standardise the numeric drivers so logistic coefficients are comparable (per 1 SD).
standardise_cols = [
    'ai_training_hours', 'num_ai_tools_used', 'ai_projects_active', 'years_using_ai',
    'ai_budget_percentage', 'log_ai_investment_per_employee',
    'regulatory_compliance_score', 'ai_risk_management_score'
]
for col in standardise_cols:
    df[f'z_{col}'] = (df[col] - df[col].mean()) / df[col].std(ddof=0)

print('Created z-score variables for regression comparability.')


Created z-score variables for regression comparability.


## Recompute headline metrics

*This notebook re-derives the five headline numbers itself so it can stand alone. The logistic models and business OLS are re-run here.*

In [5]:
# Recompute the five headline numbers here so this notebook runs on its own.
# (The logistic models and business OLS are re-run below.)

# Q1 - overall (pooled) advanced adoption rate
p_hat = df['advanced_adoption'].mean()

# Q2 - country readiness PC1 -> OLS adjusted R2
country_level = (df.groupby('country')
                   .agg(adv_adoption_rate=('advanced_adoption', 'mean'),
                        log_gdp_per_capita=('log_gdp_per_capita', 'first'),
                        internet_penetration=('internet_penetration', 'first'),
                        digital_maturity_index=('digital_maturity_index', 'first'),
                        log_ai_patent_filings_2024=('log_ai_patent_filings_2024', 'first'),
                        ai_researchers_per_million=('ai_researchers_per_million', 'first'))
                   .reset_index())
readiness_features = ['log_gdp_per_capita', 'internet_penetration', 'digital_maturity_index',
                      'log_ai_patent_filings_2024', 'ai_researchers_per_million']
X_readiness = StandardScaler().fit_transform(country_level[readiness_features])
pca = PCA(n_components=2)
country_level['readiness_pc1'] = pca.fit_transform(X_readiness)[:, 0]
if country_level[['readiness_pc1', 'digital_maturity_index']].corr().iloc[0, 1] < 0:
    country_level['readiness_pc1'] *= -1
model_country_pc1 = ols('adv_adoption_rate ~ readiness_pc1', data=country_level).fit()

# Q3 - nested logistic models (pseudo-R2)
control_terms = 'C(country) + C(industry) + C(company_size) + C(survey_year) + C(quarter)'
capability_terms = 'z_ai_training_hours + z_num_ai_tools_used + z_ai_projects_active + z_years_using_ai'
investment_terms = 'z_ai_budget_percentage + z_log_ai_investment_per_employee'
governance_terms = 'z_regulatory_compliance_score + z_ai_risk_management_score + C(data_privacy_level) + C(ai_ethics_committee)'
logit_specs = {
    'M0 Controls only': control_terms,
    'M1 Controls + Capability': control_terms + ' + ' + capability_terms,
    'M2 Controls + Investment': control_terms + ' + ' + investment_terms,
    'M3 Controls + Governance': control_terms + ' + ' + governance_terms,
    'M4 Full driver model': control_terms + ' + ' + capability_terms + ' + ' + investment_terms + ' + ' + governance_terms,
}
logit_rows = []
for name, rhs in logit_specs.items():
    res = logit('advanced_adoption ~ ' + rhs, data=df).fit(method='lbfgs', maxiter=200, disp=False)
    logit_rows.append({'model': name, 'McFadden_pseudo_R2': res.prsquared})
logit_compare = pd.DataFrame(logit_rows)

# Q4 - business OLS adjusted-R2 gain from advanced adoption (only M0/M1 needed for this metric)
ols_control_terms = control_terms
business_outcomes = {
    'productivity_change_percent': 'Productivity change (%)',
    'revenue_growth_percent': 'Revenue growth (%)',
    'cost_reduction_percent': 'Cost reduction (%)',
    'innovation_score': 'Innovation score',
    'customer_satisfaction': 'Customer satisfaction'
}
business_ols_rows = []
for outcome, label in business_outcomes.items():
    m0 = ols(f'{outcome} ~ {ols_control_terms}', data=df).fit()
    m1 = ols(f'{outcome} ~ advanced_adoption + {ols_control_terms}', data=df).fit()
    business_ols_rows.append({'outcome': label, 'delta_adj_R2_advanced': m1.rsquared_adj - m0.rsquared_adj})
business_ols_summary = pd.DataFrame(business_ols_rows)

# Q5 - Enterprise posterior mean (Beta-Binomial on latest observation per company)
size_counts = (latest_obs.groupby('company_size')['advanced_adoption']
                 .agg(x='sum', n='count')
                 .reindex(['Startup', 'SME', 'Enterprise']).reset_index())
posterior_rows = []
for _, row in size_counts.iterrows():
    a_post, b_post = 1 + int(row['x']), 1 + int(row['n']) - int(row['x'])
    posterior_rows.append({'company_size': row['company_size'], 'posterior_mean': a_post / (a_post + b_post)})
posterior_summary = pd.DataFrame(posterior_rows)

print('Headline metrics recomputed for the evidence table.')


Headline metrics recomputed for the evidence table.


In [6]:
# Final compact evidence table for report/slide writing.
p_hat = df['advanced_adoption'].mean()
final_evidence = pd.DataFrame({
    'evidence_item': [
        'Overall advanced adoption rate',
        'Country readiness PC1 OLS adjusted R2',
        'Best logistic model pseudo-R2',
        'Largest business adjusted R2 gain from advanced adoption',
        'Enterprise posterior mean advanced adoption (latest company observation)'
    ],
    'value': [
        f'{p_hat:.1%}',
        f'{model_country_pc1.rsquared_adj:.3f}',
        f'{logit_compare["McFadden_pseudo_R2"].max():.3f}',
        f'{business_ols_summary["delta_adj_R2_advanced"].max():.3f}',
        f'{posterior_summary.loc[posterior_summary["company_size"] == "Enterprise", "posterior_mean"].iloc[0]:.1%}'
    ]
})

final_evidence


,evidence_item,value
0,Overall advanced adoption rate,53.7%
1,Country readiness PC1 OLS adjusted R2,0.311
2,Best logistic model pseudo-R2,0.505
3,Largest business adjusted R2 gain from advance...,0.244
4,Enterprise posterior mean advanced adoption (l...,78.6%


## Key insights
Overall, the analysis shows five main findings.

1. AI adoption is increasing, but most firms are still not fully mature. Full adoption is still rare, at around 1%. The advanced adoption rate increased from about 45% in 2023 to about 62% in 2026, but “advanced” usually means partial adoption rather than full maturity.

2. Country readiness seems related to AI adoption. Since the readiness variables are highly correlated, we used PCA, and PC1 explains about 87% of the variance. However, with only 30 countries, this result should be seen as exploratory.

3. At the firm level, capability and investment appear to be the key drivers. They improve pseudo-R2 and reduce AIC/BIC more clearly than governance. Governance does not show a strong independent effect in the full model, with odds ratios close to 1.

4. Advanced adoption is linked to better business outcomes, especially productivity, where the adjusted R2 increases by about 0.24. However, this is only an association, not proof of causality.

5. Advanced adopters also tend to have more reskilling and lower failure rates. Job creation is positive but very small, at about 0.5 jobs per 100 employees. The Bayesian analysis also suggests that enterprises are ahead, at around 79%, while SMEs and startups are closer to 60%.
